# Section 0 — Setup & Chargement

Charge le meilleur modèle produit par le notebook 03, les features labelisées et les dates d'achat brutes pour construire les fenêtres temporelles.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import json
import warnings
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

from features import FINAL_FEATURES
from model_utils import (
    build_cluster_profile,
    compute_clustering_metrics,
    fit_kmeans,
    load_model,
    save_model,
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR    = PROJECT_ROOT / 'models'

print('Répertoires :', PROCESSED_DIR, MODELS_DIR)

In [ ]:
# Chargement des features labelisées (produit par notebook 03)
df_labeled = pd.read_parquet(PROCESSED_DIR / 'customer_features_labeled.parquet')
print(f'Clients labelisés : {len(df_labeled):,} | Colonnes : {list(df_labeled.columns)}')

# Chargement du meilleur modèle
model_files = sorted(MODELS_DIR.glob('best_clustering_*.pkl'))
assert model_files, 'Aucun modèle best_clustering_*.pkl trouvé — exécuter notebook 03 d\'abord.'
best_model_path = model_files[-1]
best_model = load_model(best_model_path)
print(f'Modèle chargé : {best_model_path.name}')

# Chargement des métadonnées pour récupérer BEST_K
meta_path = best_model_path.with_suffix('.json')
with open(meta_path) as f:
    meta = json.load(f)
BEST_K = int(meta.get('k', 5))
print(f'k optimal : {BEST_K}')

In [ ]:
# Matrice X (features scalées) depuis df_labeled
# Vérification que les FINAL_FEATURES sont présentes
missing = [f for f in FINAL_FEATURES if f not in df_labeled.columns]
assert not missing, f'Features manquantes dans df_labeled : {missing}'

X_all = df_labeled[FINAL_FEATURES].values.astype(np.float64)
assert not np.isnan(X_all).any(), 'NaN dans X_all'
print(f'X_all shape : {X_all.shape}')

# Section 1 — Création des Fenêtres Temporelles

Chaque fenêtre correspond à une coupe temporelle cumulée sur les commandes Olist.
L'objectif est de simuler ce qu'aurait donné le modèle s'il avait été entraîné à différents
moments : T1 (mi-2017), T2 (fin 2017), T3 (mi-2018), T4 = dataset complet.

Pour chaque fenêtre, on retient uniquement les **clients ayant passé au moins une commande
avant la date de coupure**.


In [ ]:
# Chargement des dates de commande depuis les features brutes (si disponibles)
# ou en rechargeant le CSV orders (fallback)
raw_path = PROCESSED_DIR / 'customer_features_raw.parquet'

if raw_path.exists():
    df_raw = pd.read_parquet(raw_path)
else:
    # Fallback : construire la date de dernier achat depuis le CSV
    orders_csv = PROJECT_ROOT / 'data' / 'olist_orders_dataset.csv'
    df_orders = pd.read_csv(orders_csv, parse_dates=['order_purchase_timestamp'],
                             usecols=['customer_id', 'order_purchase_timestamp'])
    customers_csv = PROJECT_ROOT / 'data' / 'olist_customers_dataset.csv'
    df_cust = pd.read_csv(customers_csv, usecols=['customer_id', 'customer_unique_id'])
    df_dates = df_orders.merge(df_cust, on='customer_id', how='left')
    last_purchase = (df_dates.groupby('customer_unique_id')['order_purchase_timestamp']
                     .max().reset_index(name='last_purchase_date'))
    df_raw = df_labeled.merge(last_purchase, on='customer_unique_id', how='left')
    print('Fallback : dates reconstituées depuis CSV')

# Normalisation de la colonne de date
date_col = 'last_purchase_date'
assert date_col in df_raw.columns, (
    f'Colonne {date_col!r} absente. Vérifier customer_features_raw.parquet ou les CSV.')
df_raw[date_col] = pd.to_datetime(df_raw[date_col])

# Jointure df_labeled <-> dates
df_sim = df_labeled.merge(
    df_raw[['customer_unique_id', date_col]].drop_duplicates('customer_unique_id'),
    on='customer_unique_id', how='left'
)
print(f'df_sim shape : {df_sim.shape} | nulls sur date : {df_sim[date_col].isna().sum()}')

In [ ]:
# Définition des 4 fenêtres temporelles (cumulées)
WINDOWS = {
    'T1': pd.Timestamp('2017-06-30'),
    'T2': pd.Timestamp('2017-12-31'),
    'T3': pd.Timestamp('2018-06-30'),
    'T4': df_sim[date_col].max(),  # Full dataset
}

windows_data: dict[str, pd.DataFrame] = {}
for name, cutoff in WINDOWS.items():
    mask = df_sim[date_col] <= cutoff
    windows_data[name] = df_sim[mask].copy()
    print(f'{name} (≤ {cutoff.date()}) : {mask.sum():>7,} clients')

# Section 2 — Re-clustering par Fenêtre

Pour chaque fenêtre temporelle, on réentraîne KMeans avec le même `k` que le meilleur
modèle sélectionné en notebook 03. Cela permet de mesurer comment les frontières de
clusters évoluent dans le temps.

> **Note :** On utilise les features scalées déjà présentes dans `df_labeled`.
> Dans un pipeline production, il faudrait re-scaler chaque fenêtre indépendamment
> (ou utiliser le scaler entraîné sur T4 et le appliquer aux fenêtres précédentes).


In [ ]:
window_labels: dict[str, np.ndarray] = {}
window_customer_ids: dict[str, list[str]] = {}

for name, df_w in windows_data.items():
    X_w = df_w[FINAL_FEATURES].values.astype(np.float64)
    if len(X_w) < BEST_K * 10:
        print(f'{name} : pas assez de données ({len(X_w)} < {BEST_K * 10}), ignoré.')
        continue

    km = __import__('sklearn.cluster', fromlist=['KMeans']).KMeans(
        n_clusters=BEST_K, init='k-means++', n_init=20, random_state=42
    )
    labels = km.fit_predict(X_w)
    window_labels[name] = labels
    window_customer_ids[name] = list(df_w['customer_unique_id'])

    metrics = compute_clustering_metrics(X_w, labels)
    print(f'{name} | k={BEST_K} | silhouette={metrics["silhouette"]:.4f} '
          f'| DB={metrics["davies_bouldin"]:.4f} | n={len(X_w):,}')

# Section 3 — Adjusted Rand Index (ARI) entre Fenêtres Consécutives

L'**ARI** mesure la similarité entre deux clusterings sur les mêmes clients.
Il est calculé sur l'**intersection** des clients communs entre deux fenêtres consécutives.

| ARI | Interprétation |
|-----|----------------|
| > 0.7 | Clustering très stable — même structure |
| 0.5 – 0.7 | Dérive modérée — surveiller |
| < 0.5 | Dérive significative — ré-entraînement recommandé |


In [ ]:
pairs = [('T1', 'T2'), ('T2', 'T3'), ('T3', 'T4')]
ari_scores: dict[str, float] = {}

for t_prev, t_next in pairs:
    if t_prev not in window_labels or t_next not in window_labels:
        print(f'Paire {t_prev}→{t_next} ignorée (données insuffisantes).')
        continue

    ids_prev = set(window_customer_ids[t_prev])
    ids_next = set(window_customer_ids[t_next])
    common = sorted(ids_prev & ids_next)

    if len(common) < 50:
        print(f'Paire {t_prev}→{t_next} : seulement {len(common)} clients communs.')
        continue

    prev_map = dict(zip(window_customer_ids[t_prev], window_labels[t_prev]))
    next_map = dict(zip(window_customer_ids[t_next], window_labels[t_next]))

    y_prev = np.array([prev_map[c] for c in common])
    y_next = np.array([next_map[c] for c in common])

    ari = adjusted_rand_score(y_prev, y_next)
    ari_scores[f'{t_prev}→{t_next}'] = round(ari, 4)
    print(f'ARI {t_prev}→{t_next} : {ari:.4f}  ({len(common):,} clients communs)')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels_ari = list(ari_scores.keys())
values_ari = list(ari_scores.values())
colors = ['#2ecc71' if v > 0.7 else '#f39c12' if v > 0.5 else '#e74c3c' for v in values_ari]

bars = ax.bar(labels_ari, values_ari, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(0.7, color='#2ecc71', linestyle='--', linewidth=1.2, label='Seuil stable (0.7)')
ax.axhline(0.5, color='#e74c3c', linestyle='--', linewidth=1.2, label='Seuil dérive (0.5)')
ax.set_ylim(0, 1.05)
ax.set_ylabel('ARI', fontsize=12)
ax.set_title('Adjusted Rand Index entre Fenêtres Consécutives', fontsize=13)
ax.legend(fontsize=10)
for bar, val in zip(bars, values_ari):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f'{val:.3f}',
            ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print('ARI scores :', ari_scores)

# Section 4 — Silhouette Score par Fenêtre Temporelle

Une tendance décroissante du score de silhouette indique que la structure des clusters
se dégrade au fil du temps : les clients deviennent moins bien séparés dans l'espace
de features. Une chute > 0.05 entre deux fenêtres consécutives est un signal de dérive.


In [ ]:
silhouette_by_window: dict[str, float] = {}

for name, df_w in windows_data.items():
    if name not in window_labels:
        continue
    X_w = df_w[FINAL_FEATURES].values.astype(np.float64)
    metrics = compute_clustering_metrics(X_w, window_labels[name])
    silhouette_by_window[name] = round(metrics['silhouette'], 4)

print('Silhouette par fenêtre :')
for w, sil in silhouette_by_window.items():
    print(f'  {w} : {sil:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
windows_list = list(silhouette_by_window.keys())
sil_values   = list(silhouette_by_window.values())

ax.plot(windows_list, sil_values, marker='o', linewidth=2.5,
        color='#3498db', markersize=9, label='Silhouette')
for i, (w, v) in enumerate(zip(windows_list, sil_values)):
    ax.annotate(f'{v:.4f}', (w, v), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=10)

# Seuil de dérive : chute > 0.05 par rapport au max
if sil_values:
    drift_threshold = max(sil_values) - 0.05
    ax.axhline(drift_threshold, color='#e74c3c', linestyle='--',
               linewidth=1.2, label=f'Seuil dérive (max - 0.05 = {drift_threshold:.3f})')

ax.set_ylabel('Score de Silhouette', fontsize=12)
ax.set_title('Évolution du Score de Silhouette par Fenêtre', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Section 5 — Stabilité des Profils de Clusters

Comparaison des médianes des clusters entre T3 et T4 (les deux fenêtres les plus récentes).
Un delta CLV_proxy > 20% entre T3 et T4 sur le même cluster signifie que le comportement
moyen du segment a changé — signal de ré-entraînement immédiat.


In [ ]:
profile_by_window: dict[str, pd.DataFrame] = {}
RAW_PROFILE_FEATURES = ['Monetary', 'Frequency', 'Frequency_flag',
                         'avg_delivery_delay', 'avg_review_score', 'avg_freight_ratio']
raw_cols = [c for c in RAW_PROFILE_FEATURES if c in windows_data['T4'].columns]

for name, df_w in windows_data.items():
    if name not in window_labels or not raw_cols:
        continue
    prof = build_cluster_profile(df_w, window_labels[name], raw_cols)
    profile_by_window[name] = prof
    print(f'\n--- Profil {name} ---')
    print(prof[['cluster', 'n_customers', 'CLV_proxy'] + raw_cols[:3]].to_string(index=False))

In [ ]:
# Delta CLV_proxy T3 → T4 par cluster (sur les clusters communs)
if 'T3' in profile_by_window and 'T4' in profile_by_window:
    p3 = profile_by_window['T3'].set_index('cluster')[['CLV_proxy']].rename(
        columns={'CLV_proxy': 'CLV_T3'})
    p4 = profile_by_window['T4'].set_index('cluster')[['CLV_proxy']].rename(
        columns={'CLV_proxy': 'CLV_T4'})
    delta_df = p3.join(p4, how='inner')
    delta_df['delta_pct'] = ((delta_df['CLV_T4'] - delta_df['CLV_T3'])
                              / delta_df['CLV_T3'].abs() * 100).round(2)
    print('Delta CLV_proxy T3 → T4 :')
    print(delta_df.to_string())

    alert_clusters = delta_df[delta_df['delta_pct'].abs() > 20]
    if len(alert_clusters) > 0:
        print(f'\n⚠️  {len(alert_clusters)} cluster(s) avec delta CLV > 20% : '
              f'{list(alert_clusters.index)}')
    else:
        print('\n✓ Aucun cluster avec delta CLV > 20%.')
else:
    print('T3 ou T4 manquant — comparaison impossible.')

# Section 6 — Bootstrap Variance (Stabilité Interne)

30 rééchantillonnages bootstrap de T4 (avec remise) permettent d'estimer la variance
du score de silhouette et de calculer un intervalle de confiance à 95%.

**Critère de stabilité :** CV (= std/mean) < 0.05 → le modèle est robuste.


In [ ]:
N_BOOTSTRAP = 30
bootstrap_sil: list[float] = []
rng = np.random.default_rng(42)

df_t4 = windows_data.get('T4', df_sim)
X_t4 = df_t4[FINAL_FEATURES].values.astype(np.float64)
n_t4 = len(X_t4)

print(f'Bootstrap sur T4 (n={n_t4:,}) — {N_BOOTSTRAP} itérations...')
for i in range(N_BOOTSTRAP):
    idx = rng.choice(n_t4, size=n_t4, replace=True)
    X_boot = X_t4[idx]
    km_boot = __import__('sklearn.cluster', fromlist=['KMeans']).KMeans(
        n_clusters=BEST_K, n_init=5, random_state=i)
    labels_boot = km_boot.fit_predict(X_boot)
    m = compute_clustering_metrics(X_boot, labels_boot)
    if not np.isnan(m['silhouette']):
        bootstrap_sil.append(m['silhouette'])

boot_mean = np.mean(bootstrap_sil)
boot_std  = np.std(bootstrap_sil)
boot_cv   = boot_std / boot_mean if boot_mean > 0 else np.nan
ci95_lo   = np.percentile(bootstrap_sil, 2.5)
ci95_hi   = np.percentile(bootstrap_sil, 97.5)

print(f'Mean silhouette : {boot_mean:.4f}')
print(f'Std silhouette  : {boot_std:.4f}')
print(f'CV              : {boot_cv:.4f}  ({"STABLE" if boot_cv < 0.05 else "INSTABLE"})')
print(f'IC 95%          : [{ci95_lo:.4f}, {ci95_hi:.4f}]')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(bootstrap_sil, bins=20, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(boot_mean, color='#e74c3c', linewidth=2, label=f'Moyenne = {boot_mean:.4f}')
ax.axvline(ci95_lo, color='#f39c12', linestyle='--', linewidth=1.5, label=f'IC 95% [{ci95_lo:.4f}, {ci95_hi:.4f}]')
ax.axvline(ci95_hi, color='#f39c12', linestyle='--', linewidth=1.5)
ax.set_xlabel('Score de Silhouette (bootstrap)', fontsize=12)
ax.set_ylabel('Fréquence', fontsize=12)
ax.set_title(f'Distribution Bootstrap du Score de Silhouette (n={N_BOOTSTRAP} iter.)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Section 7 — Recommandation Fréquence de Mise à Jour

Synthèse des signaux de stabilité observés dans les sections précédentes.

| Critère | Seuil | Recommandation |
|---------|-------|----------------|
| ARI T3→T4 | > 0.7 | ✅ Semestriel |
| ARI T3→T4 | 0.5 – 0.7 | ⚠️ Trimestriel |
| ARI T3→T4 | < 0.5 | 🚨 Mensuel + alerting |
| Delta CLV_proxy | > 20% sur un cluster | 🚨 Ré-entraînement immédiat |
| CV bootstrap silhouette | < 0.05 | ✅ Modèle robuste |


In [ ]:
# ── Synthèse automatique des signaux ────────────────────────────────────
signals: list[dict] = []

# Signal 1 : ARI T3→T4
ari_t3_t4 = ari_scores.get('T3→T4', None)
if ari_t3_t4 is not None:
    if ari_t3_t4 > 0.7:
        freq_ari = 'semestriel'
        status_ari = '✅ STABLE'
    elif ari_t3_t4 > 0.5:
        freq_ari = 'trimestriel'
        status_ari = '⚠️  ATTENTION'
    else:
        freq_ari = 'mensuel + alerting automatique'
        status_ari = '🚨 DÉRIVE DÉTECTÉE'
    signals.append({'critere': f'ARI T3→T4 = {ari_t3_t4:.4f}',
                    'status': status_ari, 'recommandation': freq_ari})

# Signal 2 : Delta CLV_proxy
if 'delta_df' in dir() and len(delta_df) > 0:
    max_delta = delta_df['delta_pct'].abs().max()
    if max_delta > 20:
        signals.append({'critere': f'Delta CLV max = {max_delta:.1f}%',
                        'status': '🚨 DÉRIVE PROFIL', 'recommandation': 'immédiat'})
    else:
        signals.append({'critere': f'Delta CLV max = {max_delta:.1f}%',
                        'status': '✅ STABLE', 'recommandation': 'aucun ré-entraînement urgent'})

# Signal 3 : CV Bootstrap
cv_status = '✅ ROBUSTE' if boot_cv < 0.05 else '⚠️  INSTABLE'
signals.append({'critere': f'CV Bootstrap = {boot_cv:.4f}',
                'status': cv_status,
                'recommandation': 'modèle robuste' if boot_cv < 0.05 else 'vérifier la stabilité interne'})

signals_df = pd.DataFrame(signals)
print('\n=== SYNTHÈSE DES SIGNAUX DE STABILITÉ ===')
print(signals_df.to_string(index=False))

# Recommandation finale (règle la plus conservatrice)
freq_map = {'immédiat': 0, 'mensuel + alerting automatique': 1,
            'trimestriel': 2, 'semestriel': 3,
            'aucun ré-entraînement urgent': 4, 'modèle robuste': 4}
rec_freqs = [r for r in signals_df['recommandation'] if r in freq_map]
FINAL_RECOMMENDATION = min(rec_freqs, key=lambda r: freq_map[r]) if rec_freqs else 'trimestriel'

print(f'\n>>> RECOMMANDATION FINALE : ré-entraînement {FINAL_RECOMMENDATION.upper()} <<<')

# Section 8 — Export du Rapport de Stabilité

Génère `data/processed/stability_report.json` — lu par le dashboard pour afficher
la recommandation de fréquence de mise à jour dans l'onglet Vue d'Ensemble.


In [ ]:
stability_report = {
    'generated_at': datetime.now().isoformat(),
    'best_model': best_model_path.name,
    'best_k': BEST_K,
    'windows': {w: str(cutoff.date()) for w, cutoff in WINDOWS.items()},
    'n_customers_per_window': {w: len(df_w) for w, df_w in windows_data.items()},
    'ari_scores': ari_scores,
    'silhouette_by_window': silhouette_by_window,
    'bootstrap': {
        'n_iterations': N_BOOTSTRAP,
        'mean_silhouette': round(float(boot_mean), 4),
        'std_silhouette': round(float(boot_std), 4),
        'cv': round(float(boot_cv), 4),
        'ci95': [round(float(ci95_lo), 4), round(float(ci95_hi), 4)],
    },
    'recommendation': FINAL_RECOMMENDATION,
    'signals': signals_df.to_dict(orient='records'),
}

report_path = PROCESSED_DIR / 'stability_report.json'
with open(report_path, 'w') as f:
    json.dump(stability_report, f, indent=2, default=str)

print(f'Rapport exporté : {report_path}')
print(f'Recommandation  : {FINAL_RECOMMENDATION}')

# ── Sanity check ─────────────────────────────────────────────────────────────
assert report_path.exists()
with open(report_path) as f:
    loaded = json.load(f)
assert 'recommendation' in loaded
assert 'ari_scores' in loaded
print('\n✓ Sanity check OK — stability_report.json valide.')